#### Relevant Imports

# Notebook 4 — End-to-End RAG Pipeline with LLM-Generated SQL & Validation

## What You Will Learn

This notebook closes the loop on RAG: instead of writing SQL by hand, you let the **LLM generate the query** from the user's question. A second LLM call then **validates** whether that query is actually sufficient to answer the question — before any data is fetched.

### Two-Pass Pipeline

```
User Question + Schema
        │
        ▼  (Pass 1 — Query Generation)
   LLM → SQL Query
        │
        ▼  (Pass 2 — Query Validation)
   LLM → Pass ✅ / Fail ❌
        │
        ▼  (Pass 3 — Answer)
   Execute SQL → LLM → Final Answer
```

### Topics Covered

| Stage | What Happens |
|---|---|
| **SQL Query Generation** | LLM turns a natural-language question into a valid SQLite query |
| **Prompt Engineering** | Instructions for case-insensitive search, wildcards, SQLite-specific syntax |
| **Query Validation** | Second LLM evaluates schema relevance, accuracy, and sufficiency |
| **Error Injection** | Deliberately broken queries to test the validator |
| **Complex Schemas** | Multi-table schemas with foreign keys (Invoice, Customer, InvoiceLine) |

### Skills You Will Build

- Prompt an LLM to produce raw SQL with no markdown fences or preamble
- Add a quality-gate LLM call that returns a structured Pass/Fail JSON verdict
- Test your validation logic with intentionally wrong queries
- Handle multi-table schemas in prompt context

> **Pre-requisite:** Notebooks 1–3. Requires Groq and Gemini API keys.

In [16]:
from sqlalchemy import create_engine, text
import json
from collections import defaultdict
from typing import List, Dict, Any
import csv
from hf_llm import hf_chat_completion
import dotenv
import os

**SQLite DB**  
Let us connect to the SQLite Sample Database what we have  


In [17]:
# There is an engine instance created, which can handle multiple connetions
DB_File = "Sample_2.db"

if os.path.exists (DB_File):
    sql_engine = create_engine("sqlite:///"+DB_File)
    conn_1 = sql_engine.connect ()
else:
    print ("DB Files does not exist")

**Get Query**  
Given a user prompt and table schema, LLM can generate SQL query  
The query generation requires specific instructions, so that it can be used correctly

In [18]:
# Groq / Gemini via hf_llm (falls back to local HF if no API key)
dotenv.load_dotenv ()
model_gr = "llama-3.3-70b-versatile"
model_gm = "gemini-3.5-flash-lite"

In [19]:
Q_Instr = "From the given SQL table schema, formulate SQL query which answers user's question. \
        Write query to fetch all relevant details to answer the question\
        Respond only the SQL query. No additional text, no fence" # prompt instruction for sql query gen

Schema = """
            CREATE TABLE Student_Performance (
            student_id VARCHAR(50),
            age INTEGER,
            gender VARCHAR(50),
            study_hours_per_day REAL,
            social_media_hours REAL,
            netflix_hours REAL,
            part_time_job VARCHAR(50),
            attendance_percentage REAL,
            sleep_hours REAL,
            diet_quality VARCHAR(50),
            exercise_frequency INTEGER,
            parental_education_level VARCHAR(50),
            internet_quality VARCHAR(50),
            mental_health_rating INTEGER,
            extracurricular_participation VARCHAR(50),
            exam_score REAL
          )
        """

Prompt = "On an average students watch netflix for ..." # user question that can be answered using data in the database
# Prompt = "Are the students taking enough rest?"
# Prompt = "Is there a correlation between study time and score really?"
# Prompt = "Tell me top 5 scorer who are doing a job additionally, because its quite challenging"

messages=[
    {
        "role": "system",
        "content": Q_Instr
    },

    {
        "role": "user",
        "content": "Schema :\n"+Schema+"\n Question : \n"+Prompt
    }
]
completion = hf_chat_completion(
    messages=messages,
    model=model_gr,
    # temperature=0.0
)

print (completion.choices[0].message.content)

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


>Additional instructions to fine tune the query generation  
>This will help to get precise output.

In [20]:
Q_Instr = "From the given SQL table schema, formulate SQL query which answers user's question.\
        Write query to fetch all relevant details to answer the question\
        Respond only the SQL query. No title, no introduction. Give complete query.\
        Its SQLite DB.\
        When searching text field, always convert to lower case and use wild card : %xxx% .\
        If question is not relevant to schema, say 'No relevant data'"


Schema = """
            CREATE TABLE Student_Performance (
            student_id VARCHAR(50),
            age INTEGER,
            gender VARCHAR(50),
            study_hours_per_day REAL,
            social_media_hours REAL,
            netflix_hours REAL,
            part_time_job VARCHAR(50),
            attendance_percentage REAL,
            sleep_hours REAL,
            diet_quality VARCHAR(50),
            exercise_frequency INTEGER,
            parental_education_level VARCHAR(50),
            internet_quality VARCHAR(50),
            mental_health_rating INTEGER,
            extracurricular_participation VARCHAR(50),
            exam_score REAL
          )
        """

Prompt = "Is there a correlation between study time and score really?"
# Prompt = "Tell me top 5 scorers who are doing a job additionally, because its quite challenging"
# Prompt = "What are the kind of diet habits they have?"
# Prompt = "When are the exams?"

messages=[
    {
        "role": "system",
        "content": Q_Instr
    },

    {
        "role": "user",
        "content": "Schema :\n"+Schema+"\n Question : \n"+Prompt
    }
]
completion = hf_chat_completion(
    messages=messages,
    model=model_gr,   

)

Query_String = completion.choices[0].message.content

print (Query_String)

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________


> Try with different model

In [13]:
# Invoke HF LLM for the same prompt
response = hf_chat_completion(
    messages=[
        {"role": "system","content": Q_Instr},
        {"role": "user", "content": "Schema :\n"+Schema+"\n Question : \n"+Prompt}
    ],
    model=model_gm,
)

print("HF Response :",response.choices[0].message.content)

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


HF Response : 


**Complete Pipeline**
The query string further needs to be used for extraction, provide the context from there and get the answer from LLM  
First invoke of LLM to get the SQL Query. Then next one for getting the response

In [ ]:
# Prompt = "how many students are studying more?"
# Prompt = "Who are the top 5 scorers and how is their social media, netflix habits?"
Prompt = "How much time Students spend on line in general?"
# Prompt = "students whose parents have completed High School, have scored more than 90 in exams"


Q_Instr = "From the given SQL table schema, formulate SQL query which answers user's question.\
        Write query to retrieve required and sufficient information to provide enough context for the question.\
        When searching text field, always convert to lower case and use wild card : %xxx% .\
        Respond only the SQL query. No title, no introduction, no fence. Give complete query.\
        Its SQLite DB.\
        If question is not relevant to schema, say 'No relevant data'"

messages=[
    {
        "role": "system",
        "content": Q_Instr
    },

    {
        "role": "user",
        "content": "Schema :\n"+Schema+"\n Question : \n"+Prompt
    }
]
completion = hf_chat_completion(
    messages=messages,
    model=model_gr,

)

## Get the query string
Query_String = completion.choices[0].message.content # sql query
print(Query_String)

#Fetch the data from DB
result = conn_1.execute(text(Query_String))

# Query output into JSON
rows = [row._asdict () for row in result]
json_output = json.dumps(rows,  indent=2)
# print (json_output)

R_Instr = "Using the context given, provide response to the user question or statement.\
            Context is provided as JSON information.\
            Provide a precise answer"

# R_Instr = "Using the context given, provide response to the user question or statement.\
#             Context is provided as JSON information.\
#             Provide a factual answer with elaboration"

messages=[
    {
        "role": "system",
        "content": R_Instr
    },

    {
        "role": "user",
        "content":"Context : \n"+ json_output + "Query : \n" + Prompt
    }
]
completion = hf_chat_completion(
    messages=messages,
    model=model_gr,
)

print (completion.choices[0].message.content)

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ResourceClosedError: This result object does not return rows. It has been closed automatically.

**Check Query**  
SQL query generated might be different each time, depending on the query clarity and complexity  
However, as long as there is a clear schema and right level of information in prompt the generated query would get required information  
As an additional check, there can be an evaluator introduced to check if the data being retrieved by the SQL query is sufficient or not

In [ ]:
# Instruction to Generate SQL query
Q_Instr = "From the given SQL table schema, formulate SQL query which answers user's question.\
        Write query to retrieve all information to provide enough context for the question.\
        When searching text field, always convert to lower case and use wild card : %xxx% .\
        Respond only the SQL query. No title, no introduction. Give complete query.\
        Its SQLite DB.\
        If question is not relevant to schema, say 'No relevant data'"

# Instruction to Check the Query
C_Instr = "You are given a table schema, user question and SQL query.\
          I want you to check if the SQL query can fetch enough information from the database to answer question.\
          Give me your **Step evaluation** and **Result** in JSON as 2 fields. No additional response\
          Steps: \
          * Understand the schema.\
          * Understand user question.\
          * Check if SQL query retrives relevant information from the schema to answer the question.\
          \
          Finally, Say 'Pass' / 'Fail'"

# Schema
Schema = """
            CREATE TABLE Student_Performance (
            student_id VARCHAR(50),
            age INTEGER,
            gender VARCHAR(50),
            study_hours_per_day REAL,
            social_media_hours REAL,
            netflix_hours REAL,
            part_time_job VARCHAR(50),
            attendance_percentage REAL,
            sleep_hours REAL,
            diet_quality VARCHAR(50),
            exercise_frequency INTEGER,
            parental_education_level VARCHAR(50),
            internet_quality VARCHAR(50),
            mental_health_rating INTEGER,
            extracurricular_participation VARCHAR(50),
            exam_score REAL
          )
        """

In [ ]:
# Instruction to Generate Wrong SQL query for test
Q_Instr = "From the given SQL table schema, formulate SQL query which answers user's question.\
        Write query to retrieve all information to provide enough context for the question.\
        Respond only the SQL query. No title, no introduction. No fence.\
        While generating query, make one **Deliberate** mistake, that I can use for testing"

In [ ]:
# Prompt = "How much time Students spend on line in general?"
Prompt = "how many students scored marks below 50 ?"
# Prompt = "What do you think the students with high score do differently ?"
# Prompt = "Do students tend to work hard only when they have project?"

# Make use of LLM model to generate SQL query
messages=[
    {
        "role": "system",
        "content": Q_Instr # sys prompt for sql query gen
    },

    {
        "role": "user",
        "content": "Schema :\n"+Schema+"\n Question : \n"+Prompt
    }
]
completion = hf_chat_completion(
    messages=messages,
    model=model_gr,   

)

Query_String = completion.choices[0].message.content

print ("Generated Query :\n", Query_String)

# Call LLM again to check if the Query is sufficient
messages=[
    {
        "role": "system",
        "content": C_Instr # validation prompt
    },

    {
        "role": "user",
        "content": "Schema :\n"+Schema+"\n Question : \n"+Prompt+ "\n SQL Query : \n" + Query_String
    }
]
completion = hf_chat_completion(
    messages=messages,
    model=model_gr,
)

# Check_Status = json.loads(completion.choices[0].message.content)
Check_Status = completion.choices[0].message.content

print (Check_Status)

Generated Query :
 SELECT COUNT(student_id) FROM Student_Performace WHERE exam_score < 50
```json
{
  "Step evaluation": [
    "Understand the schema: The schema contains a table named Student_Performance with relevant columns such as student_id and exam_score.",
    "Understand user question: The question asks for the number of students who scored marks below 50 in the exam.",
    "Check if SQL query retrieves relevant information: The SQL query attempts to count the number of students with exam scores less than 50 from the Student_Performance table."
  ],
  "Result": "Fail"
}
```


**Complex Schema**  
When complex schema with multiple tables are there, the query generation would become more subjective  
In this scenario, the evaluation becomes more relevant

In [ ]:
# Instruction to Generate SQL query
Q_Instr = "From the given SQL table schema, formulate SQL query which answers user's question.\
        Write query to retrieve all information to provide enough context for the question.\
        When searching text field, always convert to lower case and use wild card : %xxx% .\
        Respond only the SQL query. No title, no introduction. Give complete query.\
        Its SQLite DB.\
        If question is not relevant to schema, say 'No relevant data'"

# Instruction to Check the Query
C_Instr = "You are given a SQL schema, user question and SQL query.\
          I want you to check if the SQL query can fetch enough information from the database to answer question.\
          Give me your **Step evaluation** and **Result** in JSON as 2 fields. No additional response\
          Steps: \
          * Understand the schema.\
          * Understand user question.\
          * Check if SQL query retrives relevant information from the schema to answer the question.\
          * Check if the Query is accurate in identifying the right relation between tables and fields\
          \
          Finally, Say 'Pass' / 'Fail'"

# Schema
Schema = """
CREATE TABLE [Invoice]
(
    [InvoiceId] INTEGER  NOT NULL,
    [CustomerId] INTEGER  NOT NULL,
    [InvoiceDate] DATETIME  NOT NULL,
    [BillingAddress] NVARCHAR(70),
    [BillingCity] NVARCHAR(40),
    [BillingState] NVARCHAR(40),
    [BillingCountry] NVARCHAR(40),
    [BillingPostalCode] NVARCHAR(10),
    [Total] NUMERIC(10,2)  NOT NULL,
    CONSTRAINT [PK_Invoice] PRIMARY KEY  ([InvoiceId]),
    FOREIGN KEY ([CustomerId]) REFERENCES [Customer] ([CustomerId]) 		
);

CREATE TABLE [Customer]
(
    [CustomerId] INTEGER  NOT NULL,
    [FirstName] NVARCHAR(40)  NOT NULL,
    [LastName] NVARCHAR(20)  NOT NULL,
    [Company] NVARCHAR(80),
    [Address] NVARCHAR(70),
    [City] NVARCHAR(40),
    [State] NVARCHAR(40),
    [Country] NVARCHAR(40),
    [PostalCode] NVARCHAR(10),
    [Phone] NVARCHAR(24),
    [Fax] NVARCHAR(24),
    [Email] NVARCHAR(60)  NOT NULL,
    [SupportRepId] INTEGER,
    CONSTRAINT [PK_Customer] PRIMARY KEY  ([CustomerId]),
    FOREIGN KEY ([SupportRepId]) REFERENCES [Employee] ([EmployeeId]) 
);

CREATE TABLE [InvoiceLine]
(
    [InvoiceLineId] INTEGER  NOT NULL,
    [InvoiceId] INTEGER  NOT NULL,
    [TrackId] INTEGER  NOT NULL,
    [UnitPrice] NUMERIC(10,2)  NOT NULL,
    [Quantity] INTEGER  NOT NULL,
    CONSTRAINT [PK_InvoiceLine] PRIMARY KEY  ([InvoiceLineId]),
    FOREIGN KEY ([InvoiceId]) REFERENCES [Invoice] ([InvoiceId]), 		
    FOREIGN KEY ([TrackId]) REFERENCES [Track] ([TrackId]) 		
);
        """

In [ ]:
Prompt = "Top 5 customers by the total value from their invoice"
# Prompt = "The customer with highest total invoice value, purchased which track the most number of units?"

# Make use of LLM model to generate SQL query
messages=[
    {
        "role": "system",
        "content": Q_Instr
    },

    {
        "role": "user",
        "content": "Schema :\n"+Schema+"\n Question : \n"+Prompt
    }
]
completion = hf_chat_completion(
    messages=messages,
    model=model_gr,   

)

Query_String = completion.choices[0].message.content

print ("Generated Query :\n", Query_String)

# Call LLM again to check if the Query is sufficient
messages=[
    {
        "role": "system",
        "content": C_Instr
    },

    {
        "role": "user",
        "content": "Schema :\n"+Schema+"\n Question : \n"+Prompt+ "\n SQL Query : \n" + Query_String
    }
]
completion = hf_chat_completion(
    messages=messages,
    model=model_gr
    # temperature=0.0

)

# Check_Status = json.loads(completion.choices[0].message.content)
Check_Status = completion.choices[0].message.content

print (Check_Status)

Generated Query :
 ```sql
SELECT c.FirstName, c.LastName, SUM(i.Total) as TotalValue
FROM Customer c
JOIN Invoice i ON c.CustomerId = i.CustomerId
GROUP BY c.CustomerId, c.FirstName, c.LastName
ORDER BY TotalValue DESC
LIMIT 5;
```
```json
{
  "Step evaluation": [
    "Understand the schema: The schema consists of three tables: Invoice, Customer, and InvoiceLine. The Invoice table has a foreign key to the Customer table.",
    "Understand user question: The question asks for the top 5 customers by the total value from their invoices.",
    "Check if SQL query retrieves relevant information from the schema to answer the question: The query joins the Customer and Invoice tables on the CustomerId field and calculates the total value for each customer using the SUM function.",
    "Check if the Query is accurate in identifying the right relation between tables and fields: The query correctly joins the Customer and Invoice tables and groups the results by customer."
  ],
  "Result": "Pass"


In [ ]:
threshold = 5
user_ques = ""
schemas = """ """

for i in range(threshold):
    sql_query = schemas + user_ques
    is_valid = validate(sql_query)

    if is_valid:
        results = execute(sql_query)
        json_results = json_converter(results)
        final ans = json_results + user_ques
        break

    else:
        not_valid_reasoning = validation_model_response["Step evaluation"]
        sql_query_gen_prompt = sql_query_gen_prompt + f"this was your last query {sql_query} and this was the error in your last query: {not_valid_reasoning}. please dont regenerate the same query. undrstand the error and generate a correct queyr"




SyntaxError: invalid syntax (2291248533.py, line 2)

In [ ]:
""